# 第 6 章 · Planning（OpenAI Deep Research API）

这一节的演示价值不在"调用"，而在**它把计划作为结构化的中间步骤返回**：
`reasoning`（它怎么规划）、`web_search_call`（它实际搜了什么）、
`code_interpreter_call`（它跑了什么代码）。

⚠️ **本机跑不了**：它走 OpenAI 的 `responses` 接口 + `o3-deep-research` 模型。
本仓库 `.env` 指向 DeepSeek（没有这个接口），中转站也只有 Gemini 模型。
所以这格只在缺权限时打印说明，不产生假输出——要看调用形状直接读代码。

In [1]:
import os

from openai import OpenAI

# 【适配】书里把 key 硬编码在 client 里；这里从环境变量读
KEY = os.getenv("OPENAI_API_KEY")
BASE = os.getenv("OPENAI_API_BASE", "https://api.openai.com/v1")

system_message = """You are a professional researcher preparing a structured, data-driven report.
Focus on data-rich insights, use reliable sources, and include inline citations."""
user_query = "Research the economic impact of semaglutide on global healthcare systems."

# 本仓库的 .env 指向 DeepSeek；DeepSeek 没有 responses 接口，中转站也没有 deep-research 模型
runnable = bool(KEY) and "deepseek" not in (BASE or "").lower()

if not runnable:
    print("⚠️ 跳过实际调用：这一节需要 OpenAI Deep Research 权限（responses 接口 + o3-deep-research）")
    print(f"   当前 OPENAI_API_BASE = {BASE}")
    print("   要真跑：export OPENAI_API_KEY=<有 Deep Research 权限的 key>，并确保 base_url 指向 OpenAI。")
    print("   下面这段代码保持书的原样，可直接阅读调用形状。")
else:
    client = OpenAI(api_key=KEY, base_url=BASE)

    response = client.responses.create(
        model="o3-deep-research-2025-06-26",
        input=[
            {"role": "developer", "content": [{"type": "input_text", "text": system_message}]},
            {"role": "user", "content": [{"type": "input_text", "text": user_query}]},
        ],
        reasoning={"summary": "auto"},
        tools=[{"type": "web_search_preview"}],
    )

    final_report = response.output[-1].content[0].text
    print(final_report)

    print("--- CITATIONS ---")
    annotations = response.output[-1].content[0].annotations
    if not annotations:
        print("No annotations found in the report.")
    else:
        for i, citation in enumerate(annotations):
            print(f"Citation {i + 1}:")
            print(f"  Cited Text: {final_report[citation.start_index:citation.end_index]}")
            print(f"  Title: {citation.title}")
            print(f"  URL: {citation.url}")

    # ★ 本章新增：计划与执行作为【结构化的中间步骤】返回 —— 这就是"看得见的规划"。
    #   对比上一份：那份的计划藏在 task 描述里，你只能从最终文本里猜它有没有照做。
    print("\n--- INTERMEDIATE STEPS ---")

    reasoning_step = next((it for it in response.output if it.type == "reasoning"), None)
    if reasoning_step:
        print("\n[Reasoning / 计划摘要]")
        for part in reasoning_step.summary:
            print(f"  - {part.text}")
    else:
        print("\nNo reasoning steps found.")

    search_step = next((it for it in response.output if it.type == "web_search_call"), None)
    if search_step:
        print("\n[Web Search / 实际执行的检索]")
        print(f"  Query: '{search_step.action['query']}'  Status: {search_step.status}")
    else:
        print("\nNo web search steps found.")

    code_step = next((it for it in response.output if it.type == "code_interpreter_call"), None)
    if code_step:
        print("\n[Code Interpreter / 实际执行的代码]")
        print(f"  ```python\n  {code_step.input}\n  ```")
        print(f"  output: {code_step.output}")
    else:
        print("\nNo code execution steps found.")

⚠️ 跳过实际调用：这一节需要 OpenAI Deep Research 权限（responses 接口 + o3-deep-research）
   当前 OPENAI_API_BASE = https://api.deepseek.com/v1
   要真跑：export OPENAI_API_KEY=<有 Deep Research 权限的 key>，并确保 base_url 指向 OpenAI。
   下面这段代码保持书的原样，可直接阅读调用形状。
